# Notebook 03 â€” Object Detection & Tracking (YOLOv8)

Trains YOLOv8-small on COCO 2017, fine-tunes on custom indoor data, exports for Jetson TensorRT.

## Cell 03-01 | Mount & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, torch

BASE    = '/content/drive/MyDrive/ARGUS'
DS      = f'{BASE}/datasets'
MODELS  = f'{BASE}/models'
CKPTS   = f'{BASE}/checkpoints'
EXPORTS = f'{BASE}/exports'
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'roboflow', 'supervision'])
from ultralytics import YOLO
print(f'Ultralytics loaded | Device: {DEVICE}')

## Cell 03-02 | Download COCO 2017 Dataset

In [ ]:
import subprocess, os, zipfile, tarfile as _tarfile

def _safe_dl(url, dest, min_mb=None):
    """wget -c resume. If file exists but too small, delete and restart."""
    if os.path.exists(dest):
        mb = os.path.getsize(dest) / 1e6
        if min_mb and mb < min_mb * 0.90:
            print(f"  Partial file {os.path.basename(dest)} ({mb:.0f}/{min_mb} MB) — removing")
            os.remove(dest)
        else:
            print(f"  ✓ {os.path.basename(dest)} ({mb:.0f} MB)")
            return True
    print(f"  Downloading {os.path.basename(dest)} ...")
    r = subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url])
    return r.returncode == 0

def _verify(path):
    """Return True if zip/tar is intact, delete it and return False if corrupt."""
    try:
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad: raise ValueError(bad)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with _tarfile.open(path) as t:
                t.getmembers()
        return True
    except Exception as e:
        print(f"  Corrupt archive ({e}) — removing for clean re-download")
        os.remove(path)
        return False

def _extract(path, dest):
    print(f"  Extracting {os.path.basename(path)} ...")
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    else:
        subprocess.run(["tar", "-xf", path, "-C", dest], check=True)


COCO_DIR  = f'{DS}/coco'
COCO_FLAG = f'{COCO_DIR}/coco_done.flag'
os.makedirs(COCO_DIR, exist_ok=True)

FILES = [
    ('http://images.cocodataset.org/zips/train2017.zip',
     f'{COCO_DIR}/train2017.zip', 17_000),  # ~17 GB
    ('http://images.cocodataset.org/zips/val2017.zip',
     f'{COCO_DIR}/val2017.zip',     750),  # ~750 MB
    ('http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
     f'{COCO_DIR}/annotations.zip',  240),  # ~240 MB
]

if os.path.exists(COCO_FLAG):
    print('✓ COCO 2017 already extracted')
else:
    all_ok = True
    for url, dest, min_mb in FILES:
        ok = _safe_dl(url, dest, min_mb=min_mb)
        if not ok: all_ok = False; continue
        if not _verify(dest):
            print(f'Re-run to re-download corrupt {os.path.basename(dest)}')
            all_ok = False; continue
        _extract(dest, COCO_DIR)
    if all_ok:
        open(COCO_FLAG, 'w').close()
        print('✅ COCO 2017 ready')
    else:
        print('⚠ Some files incomplete — re-run this cell to resume')
os.system(f'du -sh {COCO_DIR}')

## Cell 03-03 | Create COCO YAML Config

In [ ]:
import yaml

COCO_YAML   = f'{DS}/coco/coco.yaml'
coco_config = {
    'path': f'{DS}/coco', 'train': 'train2017', 'val': 'val2017', 'nc': 80,
    'names': [
        'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
        'traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat',
        'dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack',
        'umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball',
        'kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket',
        'bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple',
        'sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair',
        'couch','potted plant','bed','dining table','toilet','tv','laptop','mouse','remote',
        'keyboard','cell phone','microwave','oven','toaster','sink','refrigerator','book',
        'clock','vase','scissors','teddy bear','hair drier','toothbrush'
    ]
}
with open(COCO_YAML, 'w') as f: yaml.dump(coco_config, f, default_flow_style=False)
print(f'COCO YAML written: {COCO_YAML}')

## Cell 03-04 | Train YOLOv8-Small on COCO

In [ ]:
from ultralytics import YOLO
import shutil, os

COCO_RUN   = f'{CKPTS}/yolov8/coco_finetune'
LAST_PT    = f'{COCO_RUN}/weights/last.pt'
BEST_PT    = f'{COCO_RUN}/weights/best.pt'
SAVED_COCO = f'{MODELS}/detection/yolov8s_coco.pt'

if os.path.exists(SAVED_COCO):
    print(f'COCO pre-train already done: {SAVED_COCO}')
else:
    if os.path.exists(LAST_PT):
        print(f'Resuming COCO pre-train from last checkpoint: {LAST_PT}')
        model = YOLO(LAST_PT)
        results = model.train(resume=True)
    else:
        print('Starting COCO pre-train from scratch')
        model = YOLO('yolov8s.pt')
        results = model.train(
            data='coco.yaml', epochs=50, imgsz=640, batch=32, workers=8, device=0,
            project=f'{CKPTS}/yolov8', name='coco_finetune', pretrained=True,
            optimizer='AdamW', lr0=0.001, lrf=0.01, momentum=0.937, weight_decay=0.0005,
            warmup_epochs=3, cos_lr=True, amp=True, save=True, save_period=5,
            cache=False, val=True, plots=True, exist_ok=True,
        )
    shutil.copy(BEST_PT, SAVED_COCO)
    print(f'COCO pre-trained weights saved: {SAVED_COCO}')


## Cell 03-05 | Fine-Tune on Custom Indoor Data (Roboflow)

In [ ]:
CUSTOM_DET  = f'{DS}/custom_detection'
CUSTOM_YAML = f'{CUSTOM_DET}/data.yaml'
CUSTOM_RUN  = f'{CKPTS}/yolov8/custom_indoor'
LAST_PT     = f'{CUSTOM_RUN}/weights/last.pt'
SAVED_FINAL = f'{MODELS}/detection/yolov8s_argus_final.pt'

if os.path.exists(SAVED_FINAL):
    print(f'Custom fine-tune already done: {SAVED_FINAL}')
elif not os.path.exists(CUSTOM_YAML):
    print('No custom detection data found — skipping fine-tune.')
    print(f'Upload YOLOv8-format dataset to: {CUSTOM_DET}')
    print('  data.yaml + train/images + train/labels + valid/images + valid/labels')
    # Fall back to COCO pre-trained as final model
    import shutil
    shutil.copy(f'{MODELS}/detection/yolov8s_coco.pt', SAVED_FINAL)
    print(f'Using COCO pre-trained as final model: {SAVED_FINAL}')
else:
    if os.path.exists(LAST_PT):
        print(f'Resuming custom fine-tune from: {LAST_PT}')
        model_ft = YOLO(LAST_PT)
        results_ft = model_ft.train(resume=True)
    else:
        print('Starting custom fine-tune from scratch')
        model_ft = YOLO(f'{MODELS}/detection/yolov8s_coco.pt')
        results_ft = model_ft.train(
            data=CUSTOM_YAML, epochs=100, imgsz=640, batch=32, device=0,
            project=f'{CKPTS}/yolov8', name='custom_indoor',
            lr0=0.0001, pretrained=True, amp=True, save=True, save_period=5,
            exist_ok=True,
        )
    import shutil
    shutil.copy(f'{CUSTOM_RUN}/weights/best.pt', SAVED_FINAL)
    print(f'Custom fine-tuned model saved: {SAVED_FINAL}')


## Cell 03-06 | Export YOLOv8 to TensorRT

In [ ]:
from ultralytics import YOLO
import shutil

model_final = YOLO(f'{MODELS}/detection/yolov8s_argus_final.pt')
model_final.export(format='onnx', imgsz=640, half=True, simplify=True, opset=17)

shutil.copy(
    f'{MODELS}/detection/yolov8s_argus_final.onnx',
    f'{EXPORTS}/tensorrt/yolov8s_argus_640.onnx'
)
print(f'ONNX exported: {EXPORTS}/tensorrt/yolov8s_argus_640.onnx')
print('On Jetson: yolo export model=yolov8s_argus_final.pt format=engine imgsz=640 half=True')